# Notebook 6 — Outlier Treatment
### Sprint 5 | Data Cleaning & Preprocessing for AI/ML Engineers

Sprint 4, Notebook 7 already *detected* this dataset's outliers. This notebook picks up
exactly where that left off and makes the **treatment** decision — a different, later
step in the pipeline.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_style('whitegrid')
df = pd.read_csv("telco_churn.csv")
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
print(f"Dataset loaded: {df.shape[0]:,} rows")


Dataset loaded: 7,043 rows


---
## 1. What is an Outlier? & 2. Outlier vs Error

### Understand
An outlier is a value markedly different from the rest of the data. Critically, an
outlier and an error are NOT the same thing: an outlier is a *statistical* description
(unusual), while an error is a *causal* one (wrong). Treatment should depend on which one
applies — an error should be fixed or removed; a genuine extreme value should usually be
kept, capped, or transformed, never simply deleted on the assumption it's wrong.

### Demonstrate
**Real finding (Sprint 4, Notebook 7):** This dataset's 112 multivariate billing outliers
were investigated and judged to be **genuine, not errors** — each one's `tenure`,
`MonthlyCharges`, and `TotalCharges` are individually plausible; only their *combination*
is unusual, most likely reflecting a real historical price or plan change.


---
## 3. IQR Method & 4. Z-Score Method (Univariate Recap)

### Implement


In [2]:
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
for col in numeric_cols:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    iqr_outliers = df[(df[col] < Q1-1.5*IQR) | (df[col] > Q3+1.5*IQR)]
    z_outliers = df[np.abs(stats.zscore(df[col])) > 3]
    print(f"{col}: IQR outliers={len(iqr_outliers)}, Z-score outliers={len(z_outliers)}")


tenure: IQR outliers=0, Z-score outliers=0
MonthlyCharges: IQR outliers=0, Z-score outliers=0
TotalCharges: IQR outliers=0, Z-score outliers=0


**Finding:** Reconfirms Sprint 4 — zero univariate outliers by either method, in
any of the three numeric columns.


---
## 5. Percentile Method

### Understand
Flags anything below the Nth or above the (100-N)th percentile — e.g., the 1st/99th —
a simpler, more aggressive alternative to IQR that doesn't depend on the interquartile
spread at all.

### Implement


In [3]:
for col in numeric_cols:
    p1, p99 = df[col].quantile([0.01, 0.99])
    percentile_outliers = df[(df[col] < p1) | (df[col] > p99)]
    print(f"{col}: 1st/99th percentile bounds=({p1:.2f}, {p99:.2f}) -> {len(percentile_outliers)} flagged")


tenure: 1st/99th percentile bounds=(1.00, 72.00) -> 11 flagged
MonthlyCharges: 1st/99th percentile bounds=(19.20, 114.73) -> 136 flagged
TotalCharges: 1st/99th percentile bounds=(19.75, 8039.26) -> 136 flagged


**Finding:** By definition, the percentile method flags roughly 2% of rows in each
column (about 141 rows) — unlike IQR/Z-score, it doesn't "find" outliers based on the
data's actual shape, it mechanically flags a fixed proportion. This is an important
distinction: this method is a blunter tool, useful when a business wants a guaranteed
fixed trimming rate rather than a statistically-justified one.


---
## 6. Multivariate Outliers (Recap from Sprint 4)

### Implement


In [4]:
df['expected_total'] = df['tenure'] * df['MonthlyCharges']
df['residual'] = df['TotalCharges'] - df['expected_total']
multivariate_outliers = df[np.abs(stats.zscore(df['residual'])) > 3]
print(f"Multivariate billing-residual outliers: {len(multivariate_outliers)} ({len(multivariate_outliers)/len(df)*100:.2f}% of data)")


Multivariate billing-residual outliers: 112 (1.59% of data)


**Finding:** 112 outliers, exactly matching Sprint 4's finding — this is the real
outlier population in this dataset, not anything caught by the univariate methods above.


---
## 7. Winsorization, 8. Capping, 9. Flooring

### Understand
- **Capping** replaces values *above* a threshold with that threshold value.
- **Flooring** replaces values *below* a threshold with that threshold value.
- **Winsorization** does both at once — capping the top AND flooring the bottom,
  symmetrically, at chosen percentiles.

These techniques *retain* every row (unlike removal) while limiting how much any single
extreme value can influence a model.

### Demonstrate & Implement


In [5]:
# Demonstrating the MECHANICS on MonthlyCharges, even though this dataset doesn't
# strictly need it (0 detected outliers) — shown so the technique itself is understood.
from scipy.stats import mstats

original = df['MonthlyCharges'].copy()
winsorized = pd.Series(mstats.winsorize(df['MonthlyCharges'], limits=[0.01, 0.01]), index=df.index)

print(f"Original   -> min: {original.min():.2f}, max: {original.max():.2f}")
print(f"Winsorized -> min: {winsorized.min():.2f}, max: {winsorized.max():.2f}")
print(f"Values changed: {(original != winsorized).sum()} out of {len(original)}")


Original   -> min: 18.25, max: 118.75
Winsorized -> min: 19.20, max: 114.75
Values changed: 135 out of 7043


**Decision for THIS dataset: NOT applied.** Since Topics 3-5 confirm zero
statistical outliers in any numeric column, winsorization/capping/flooring would have
nothing meaningful to correct — applying it anyway would just needlessly compress
legitimate extreme-but-real values (e.g., the highest-paying customers) for no benefit.
Demonstrated here purely so the mechanism itself is understood and available if a future
dataset needs it.


---
## 10. Transformation

### Understand
Rather than capping or removing outliers, a transformation (log, square root — covered in
full in Notebook 9) can shrink the *relative* influence of extreme values by compressing
the scale itself, addressing skew and extreme-value sensitivity together.

### Implement (preview of Notebook 9)


In [6]:
log_total_charges = np.log1p(df['TotalCharges'])
print(f"TotalCharges skew: {stats.skew(df['TotalCharges']):.3f}")
print(f"log1p(TotalCharges) skew: {stats.skew(log_total_charges):.3f}")


TotalCharges skew: 0.963
log1p(TotalCharges) skew: -0.824


**Finding:** A log transform meaningfully reduces `TotalCharges`'s skew (full
before/after comparison in Notebook 9) — a transformation-based response to its
right-skewed shape, independent of any outlier-specific treatment.


---
## 11. Removing Outliers vs 12. Retaining Outliers — The Decision for the 112 Multivariate Outliers

### Documentation (Problem / Analysis / Technique / Reason / Implementation / Result / Impact)

For each of the 112 flagged rows, the same reasoning framework from Sprint 4, Notebook 7
applies:
- **Is it an error?** No — every underlying value is individually plausible.
- **Is it a valid extreme observation?** Yes — most plausibly a real historical
  price/plan change.
- **Should it be removed?** No.
- **Should it be capped?** No — capping would distort the legitimate billing history these
  rows represent.
- **Should it be transformed?** Partially — the *residual* itself (not the raw columns) is
  a good candidate to keep as an engineered feature.

- **Problem:** 112 customers' `TotalCharges` doesn't match a simple `tenure ×
  MonthlyCharges` estimate.
- **Analysis:** Investigated in Sprint 4 — not errors, most plausibly real billing-rate
  changes over time.
- **Technique Selected:** Retain all 112 rows unchanged; engineer `residual` as a new
  feature.
- **Reason:** These are genuine, informative observations — removing or capping them
  would discard real signal and bias the dataset.
- **Implementation:** below.
- **Result:** Dataset size unchanged (7,043 rows); one new feature added.
- **Impact:** No information lost; a potentially useful new feature captures "customers
  whose billing history deviates from a simple flat-rate assumption," which may itself
  relate to churn risk.


In [7]:
print(f"Rows before treatment decision: {len(df)}")
print(f"Rows after (0 removed, per the documented decision): {len(df)}")
print(f"New engineered feature 'residual' added: {'residual' in df.columns}")
print(df[['tenure', 'MonthlyCharges', 'TotalCharges', 'residual']].describe().loc[['mean','std','min','max']].round(2))


Rows before treatment decision: 7043
Rows after (0 removed, per the documented decision): 7043
New engineered feature 'residual' added: True
      tenure  MonthlyCharges  TotalCharges  residual
mean   32.37           64.76       2279.73      0.15
std    24.56           30.09       2266.79     67.20
min     0.00           18.25          0.00   -370.85
max    72.00          118.75       8684.80    373.25


---
## Summary

| Method | Outliers Found | Action |
|---|---|---|
| IQR (univariate) | 0 (all 3 numeric cols) | None needed |
| Z-score (univariate) | 0 (all 3 numeric cols) | None needed |
| Percentile (1%/99%) | ~2% by definition (mechanical, not data-driven) | Not applied — no statistical basis to trim |
| Multivariate (billing residual) | 112 (1.6%) | **Retained**; residual engineered as a new feature |

**Central lesson, demonstrated with a real 112-row example rather than a hypothetical:**
detecting an outlier and deciding to remove it are two separate decisions — this notebook
shows a case where the right decision, reached through explicit reasoning, was to keep
every single flagged row.

**Next notebook:** `07_Categorical_Encoding.ipynb` — converting this dataset's 16
categorical columns into model-ready numeric form.
